# Train Test Split

The aim for this notebook is to split the data into two different sets. The data from the 2024 season is used as the training data set and the data from the 2025 season will be used as the testing data set. The split is by season instead of randoming splitting the data to prevent data from the later season leaking into the training data.

In [5]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("../data/features.csv")
pd.set_option("display.max_columns", None)
print(df.columns.tolist())
df.head(20)

['id', 'season', 'homeTeam', 'awayTeam', 'homePregameElo', 'awayPregameElo', 'capacity', 'elevation', 'grass', 'dome', 'conferenceGame', 'temperature_c', 'precip_mm', 'windspeed_kmh', 'week', 'elo_diff', 'abs_elo_diff', 'combined_elo', 'away_is_fcs', 'away_is_power', 'home_is_power', 'rivalry_game', 'home_streak', 'home_win_pct_before', 'home_won_last_game', 'away_streak', 'away_win_pct_before', 'away_won_last_game', 'kickoff_hour', 'day_of_week', 'is_saturday', 'night_game', 'early_season', 'late_season', 'stadium_age', 'away_travel_km', 'away_travel_missing', 'same_state_opponent', 'homeConference', 'high_attendance']


,id,season,homeTeam,awayTeam,homePregameElo,awayPregameElo,capacity,elevation,grass,dome,conferenceGame,temperature_c,precip_mm,windspeed_kmh,week,elo_diff,abs_elo_diff,combined_elo,away_is_fcs,away_is_power,home_is_power,rivalry_game,home_streak,home_win_pct_before,home_won_last_game,away_streak,away_win_pct_before,away_won_last_game,kickoff_hour,day_of_week,is_saturday,night_game,early_season,late_season,stadium_age,away_travel_km,away_travel_missing,same_state_opponent,homeConference,high_attendance
0,401643697,2024,New Mexico,Montana State,1234.0,1480.0,39224.0,1554.471924,0,0,0,30.6,0.0,2.6,1,-246.0,246.0,2714.0,1,0,0,0,0,NaN,NaN,0,NaN,NaN,14.000000,5,1,0,1,0,64.0,NaN,1,0,Mountain West,0
1,401643696,2024,Nevada,SMU,1222.0,1690.0,27000.0,1406.576050,0,0,0,15.1,0.0,11.7,1,-468.0,468.0,2912.0,0,1,0,0,0,NaN,NaN,0,NaN,NaN,17.000000,5,1,0,1,0,58.0,2190.093084,0,0,Mountain West,0
2,401643858,2024,Hawai'i,Delaware State,1227.0,1480.0,15194.0,20.000000,0,0,0,26.8,0.1,33.0,1,-253.0,253.0,2707.0,1,0,0,0,0,NaN,NaN,0,NaN,NaN,17.983333,5,1,0,1,0,9.0,NaN,1,0,Mountain West,0
3,401628458,2024,Rutgers,Howard,1460.0,1480.0,52454.0,19.000000,0,0,0,23.4,0.0,16.1,1,-20.0,20.0,2940.0,1,0,1,0,0,NaN,NaN,0,NaN,NaN,18.000000,3,0,1,1,0,30.0,NaN,1,0,Big Ten,1
4,401644727,2024,Buffalo,Lafayette,1357.0,1480.0,30270.0,181.575974,0,0,0,21.9,0.0,11.5,1,-123.0,123.0,2837.0,1,0,0,0,0,NaN,NaN,0,NaN,NaN,19.000000,3,0,1,1,0,31.0,NaN,1,0,Mid-American,0
5,401644730,2024,Central Michigan,Central Connecticut,1333.0,1480.0,30255.0,240.900772,0,0,0,25.1,0.0,4.0,1,-147.0,147.0,2813.0,1,0,0,0,0,NaN,NaN,0,NaN,NaN,19.000000,3,0,1,1,0,52.0,NaN,1,0,Mid-American,0
6,401644729,2024,Bowling Green,Fordham,1431.0,1480.0,24000.0,205.905746,0,0,0,27.1,0.0,13.8,1,-49.0,49.0,2911.0,1,0,0,0,0,NaN,NaN,0,NaN,NaN,19.000000,3,0,1,1,0,58.0,NaN,1,0,Mid-American,0
7,401634299,2024,NC State,Western Carolina,1606.0,1480.0,58000.0,117.008995,1,0,0,30.7,0.0,4.7,1,126.0,126.0,3086.0,1,0,1,0,0,NaN,NaN,0,NaN,NaN,19.000000,3,0,1,1,0,58.0,NaN,1,0,ACC,1
8,401628582,2024,UCF,New Hampshire,1578.0,1480.0,44206.0,20.941137,1,0,0,28.4,0.0,8.0,1,98.0,98.0,3058.0,1,0,1,0,0,NaN,NaN,0,NaN,NaN,19.000000,3,0,1,1,0,17.0,NaN,1,0,Big 12,1
9,401628581,2024,Wake Forest,North Carolina A&T,1431.0,1480.0,31500.0,283.548584,0,0,0,29.1,0.0,12.0,1,-49.0,49.0,2911.0,1,0,1,0,0,NaN,NaN,0,NaN,NaN,19.000000,3,0,1,1,0,56.0,NaN,1,0,ACC,1


The following code block first defines which features will be used for this model. It takes all the columns and drops `id`, `season`, and `high_attendance`. The `id` column is dropped because it is just an identifier and has no relevance to predicting attendance. The `season` column is dropped because of how the data is split: each data set will contain only one season, so the column would hold the same value in every row and would not help the model determine attendance. It is still used to define the split itself. The `high_attendance` column is dropped because it is the label. This code block prints the features that are used as well as how many there are.

In [6]:
feature_list = df.drop(columns=['id','season', 'high_attendance']).columns.tolist()
X = df[feature_list]
y=df['high_attendance']

print(feature_list)
print(len(feature_list))


['homeTeam', 'awayTeam', 'homePregameElo', 'awayPregameElo', 'capacity', 'elevation', 'grass', 'dome', 'conferenceGame', 'temperature_c', 'precip_mm', 'windspeed_kmh', 'week', 'elo_diff', 'abs_elo_diff', 'combined_elo', 'away_is_fcs', 'away_is_power', 'home_is_power', 'rivalry_game', 'home_streak', 'home_win_pct_before', 'home_won_last_game', 'away_streak', 'away_win_pct_before', 'away_won_last_game', 'kickoff_hour', 'day_of_week', 'is_saturday', 'night_game', 'early_season', 'late_season', 'stadium_age', 'away_travel_km', 'away_travel_missing', 'same_state_opponent', 'homeConference']
37


This code block splits the data by season. All 2024 games make up the training set and all 2025 games form the testing set, with `X` holding the features and `y` holding the `high_attendance` label. A random split was not used because it would mix 2025 games into training. The printed shapes confirm the number of rows and features in each set, the season checks confirm each set contains only its intended season, and the value counts show the balance of high and low attendance games in each set.

In [7]:
#Time-based split of the data into training and testing sets
X_train = X[df['season'] == 2024].copy()
X_test = X[df['season'] == 2025].copy()
y_train = y[df['season'] == 2024]
y_test = y[df['season'] ==2025]

print(X_train.shape)
print(X_test.shape)

print(df.loc[X_train.index, 'season'].unique())
print(df.loc[X_test.index, 'season'].unique())
print(y_train.value_counts())
print(y_test.value_counts())

(815, 37)
(858, 37)
[2024]
[2025]
high_attendance
0    448
1    367
Name: count, dtype: int64
high_attendance
0    483
1    375
Name: count, dtype: int64


The code block below saves the training and testing data sets as CSV files and places them into the data folder.

In [8]:
X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)